In [ ]:
# Cell 1: Cài đặt thư viện (giữ nguyên)
!pip install transformers[torch] datasets accelerate evaluate trl peft bitsandbytes rouge_score scipy tqdm -q
!pip install protobuf==4.25.3 -q

In [ ]:
# Cell 3: Cài đặt trl phiên bản phù hợp (giữ nguyên)
!pip install trl==0.11.3

In [3]:
# Cell 2: Import và load dataset (giữ nguyên)
import torch
import warnings
import pandas as pd
from datasets import load_dataset, Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq
)
from peft import PeftModel
from trl import PPOTrainer, PPOConfig
from trl.models.modeling_value_head import AutoModelForSeq2SeqLMWithValueHead
import evaluate
import numpy as np
from tqdm import tqdm

warnings.filterwarnings("ignore")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Đang sử dụng thiết bị: {device}")

MODEL_NAME = "VietAI/vit5-base"
MAX_SAMPLES = 20000
MAX_LENGTH = 512

OUTPUT_SFT_DIR = "sft_adapter_summarization"
OUTPUT_PPO_DIR = "ppo_adapter_summarization"

dataset = load_dataset("nam194/vietnews", split=f"train[:{MAX_SAMPLES}]")

def preprocess_data(example):
    example["input_text"] = "tóm tắt: " + example["article"]
    example["target_text"] = example["abstract"]
    return example

dataset = dataset.map(
    preprocess_data,
    remove_columns=["guid", "title", "abstract", "article"]
)

dataset = dataset.shuffle(seed=42)
split_datasets = dataset.train_test_split(test_size=0.1, seed=42)
ppo_run_dataset = split_datasets["train"]

print(f"Đã tải {len(ppo_run_dataset)} mẫu để chạy PPO.")

Đang sử dụng thiết bị: cuda


README.md:   0%|          | 0.00/748 [00:00<?, ?B/s]

data/train-00000-of-00001-84acb79f6c6547(…):   0%|          | 0.00/170M [00:00<?, ?B/s]

data/validation-00000-of-00001-210cc51bf(…):   0%|          | 0.00/38.3M [00:00<?, ?B/s]

data/test-00000-of-00001-123f98d55067eb7(…):   0%|          | 0.00/38.8M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/99134 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/22184 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/22498 [00:00<?, ? examples/s]

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

Đã tải 18000 mẫu để chạy PPO.


In [4]:
# Cell 4: Giải nén mô hình SFT (giữ nguyên)
print("Đang giải nén sft_adapter_summary_archive.zip...")
!unzip -o -q sft_adapter_summary_archive.zip

import os
import shutil

if not os.path.exists("sft_adapter_summarization"):
    os.makedirs("sft_adapter_summarization")

files_to_move = [
    "adapter_config.json",
    "adapter_model.safetensors",
    "tokenizer.json",
    "tokenizer_config.json",
    "spiece.model",
    "special_tokens_map.json",
    "training_args.bin",
    "README.md"
]

for file_name in files_to_move:
    source_path = os.path.join(".", file_name)
    destination_path = os.path.join("sft_adapter_summarization", file_name)
    if os.path.exists(source_path) and not os.path.exists(destination_path):
        shutil.move(source_path, destination_path)

print("Giải nén và sắp xếp tệp hoàn tất. Thư mục 'sft_adapter_summarization' đã sẵn sàng.")

Đang giải nén sft_adapter_summary_archive.zip...
Giải nén và sắp xếp tệp hoàn tất. Thư mục 'sft_adapter_summarization' đã sẵn sàng.


In [24]:
# Cell 5: Chuẩn bị huấn luyện PPO - ĐÃ CẢI TIẾN (Thang 100 Tuyến tính)
import torch
import evaluate
import numpy as np
from transformers import BitsAndBytesConfig, AutoModelForSeq2SeqLM, AutoTokenizer
from trl import PPOConfig, PPOTrainer, AutoModelForSeq2SeqLMWithValueHead
from peft import PeftModel

print("--- Chuẩn bị Huấn luyện PPO ---")

# --- 1. Cấu hình Quantization ---
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_enable_fp32_cpu_offload=True
)
print("--- Quantization config đã tạo ---")

# --- 2. Cấu hình PPO (Đã có max_grad_norm) ---
# PPO Config với max_grad_norm để ổn định
ppo_config = PPOConfig(
    learning_rate=1.41e-5,
    batch_size=16,
    mini_batch_size=4,
    gradient_accumulation_steps=4,

    ppo_epochs=1,
    cliprange=0.1,
    cliprange_value=0.1,
    gamma=0.99,
    lam=0.95,
    max_grad_norm=1.0,  # ✅ Rất quan trọng để chống bùng nổ gradient

    remove_unused_columns=False,
    log_with=None,
    tracker_project_name=None,
    optimize_cuda_cache=True,
    is_encoder_decoder=True,
    seed=42,
)
print("--- PPOConfig đã tạo (với max_grad_norm=1.0) ---")

# --- 3. Tải Model PPO và Model Tham chiếu ---
# Giả định MODEL_NAME và OUTPUT_SFT_DIR đã được định nghĩa ở cell trước
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

ppo_peft_model = PeftModel.from_pretrained(base_model, OUTPUT_SFT_DIR, is_trainable=True)
ppo_model = AutoModelForSeq2SeqLMWithValueHead(ppo_peft_model)
ppo_model.is_peft_model = True

ref_base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=quantization_config,
    device_map="auto",
    torch_dtype=torch.float16
)

ref_peft_model = PeftModel.from_pretrained(ref_base_model, OUTPUT_SFT_DIR, is_trainable=False)
ref_model = AutoModelForSeq2SeqLMWithValueHead(ref_peft_model)
ref_model.is_peft_model = True
for param in ref_model.parameters():
    param.requires_grad = False

print("--- Model PPO và Model Tham chiếu đã được tải ---")

# --- 4. Tải Tokenizer ---
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
print("--- Tokenizer đã được tải ---")


# --- 5. HÀM REWARD ĐÃ CẢI TIẾN: Ổn định (Thang 100 tuyến tính) ---
try:
    rouge_metric = evaluate.load("rouge")
    print("--- Metric ROUGE đã được tải thành công ---")
except Exception as e:
    print(f"LỖI: Không thể tải ROUGE metric: {e}")
    # Có thể bạn muốn dừng lại ở đây nếu metric là bắt buộc
    # raise e

def compute_rouge_reward(predictions, references):
    """
    HÀM REWARD ĐÃ SỬA: Ổn định (Smooth) và Tuyến tính (Linear)

    Tính ROUGE-L F-measure làm phần thưởng, sử dụng thang 100 tuyến tính (score * 100).
    Hàm này mượt mà, không có "vách đá" (cliffs) và dễ diễn giải.
    """
    try:
        # --- 1. Lọc các đầu vào không hợp lệ (Rất quan trọng) ---
        valid_indices = []
        valid_predictions = []
        valid_references = []

        for i, (pred, ref) in enumerate(zip(predictions, references)):
            # Đảm bảo cả prediction và reference đều không rỗng
            if pred and pred.strip() and ref and ref.strip():
                valid_indices.append(i)
                valid_predictions.append(pred)
                valid_references.append(ref)

        # Nếu không có cặp nào hợp lệ, trả về 0 cho tất cả
        if not valid_predictions:
            print("Cảnh báo: Tất cả các tóm tắt đều rỗng. Trả về reward 0.")
            return [0.0] * len(predictions)

        # --- 2. Tính toán ROUGE ---
        rouge_scores = rouge_metric.compute(
            predictions=valid_predictions,
            references=valid_references,
            use_aggregator=False,  # Lấy điểm cho từng mẫu
            use_stemmer=True       # Thêm use_stemmer để so khớp tốt hơn
        )

        # Ưu tiên ROUGE-L, fallback sang ROUGE-Lsum
        if 'rougeL' in rouge_scores:
            key = 'rougeL'
        elif 'rougeLsum' in rouge_scores:
            key = 'rougeLsum'
        else:
            key = 'rouge1' # Fallback cuối cùng

        valid_scores = rouge_scores[key]

        # --- 3. Tính Reward với thang 100 tuyến tính (ỔN ĐỊNH) ---
        full_rewards = [0.0] * len(predictions) # Mảng reward đầy đủ

        for idx, score in zip(valid_indices, valid_scores):
            # Hàm tuyến tính: score * 100
            # (ví dụ: ROUGE 0.5 -> 50 điểm; ROUGE 0.3 -> 30 điểm)
            reward = score * 100.0
            full_rewards[idx] = min(reward, 100.0) # Đảm bảo không > 100

        # --- 4. In thống kê để Debug ---
        mean_score = np.mean(valid_scores)
        max_score = np.max(valid_scores)
        min_score = np.min(valid_scores)
        # Dùng `or [0.0]` để tránh lỗi mean của list rỗng
        mean_reward = np.mean([r for r in full_rewards if r > 0] or [0.0])

        print(f"ROUGE {key} - Mean: {mean_score:.3f}, Max: {max_score:.3f}, Min: {min_score:.3f}")
        print(f"Reward (Linear 100) - Mean: {mean_reward:.1f}, Max: {np.max(full_rewards):.1f}")

        return full_rewards

    except Exception as e:
        print(f"Lỗi nghiêm trọng khi tính ROUGE: {e}. Trả về reward 0.")
        return [0.0] * len(predictions)

# --- 6. Khởi tạo PPO Trainer ---
# Giả định ppo_run_dataset đã được định nghĩa ở cell trước
ppo_trainer = PPOTrainer(
    config=ppo_config,
    model=ppo_model,
    ref_model=ref_model,
    tokenizer=tokenizer,
    dataset=ppo_run_dataset,
    data_collator=None # data_collator=None là đúng vì chúng ta dùng dataset
)

print("\n--- Khởi tạo PPO Trainer và Reward Function (Linear 100) hoàn tất ---")
print("--- Sẵn sàng để huấn luyện! ---")

--- Chuẩn bị Huấn luyện PPO ---
--- Quantization config đã tạo ---
--- PPOConfig đã tạo (với max_grad_norm=1.0) ---
--- Model PPO và Model Tham chiếu đã được tải ---
--- Tokenizer đã được tải ---
--- Metric ROUGE đã được tải thành công ---

--- Khởi tạo PPO Trainer và Reward Function (Linear 100) hoàn tất ---
--- Sẵn sàng để huấn luyện! ---


In [25]:
# Cell 6: Huấn luyện PPO - GIẢM OUTPUT HIỂN THỊ
print(f"Bắt đầu huấn luyện PPO với {len(ppo_run_dataset)} mẫu...")

# Generation config
generation_kwargs = {
    "max_new_tokens": 48,
    "num_beams": 1,
    "do_sample": False,
    "pad_token_id": tokenizer.pad_token_id,
    "eos_token_id": tokenizer.eos_token_id,
    "early_stopping": True,
}

total_batches = len(ppo_trainer.dataloader)
pbar = tqdm(ppo_trainer.dataloader, desc="PPO Training", total=total_batches)

success_count = 0
cumulative_rewards = []

for batch_idx, batch in enumerate(pbar):
    try:
        prompt_texts = batch['input_text']
        reference_texts = batch['target_text']

        # Tokenize
        prompt_tensors = tokenizer(
            prompt_texts,
            padding=True,
            truncation=True,
            max_length=192,
            return_tensors="pt"
        ).to(ppo_trainer.current_device)

        # Generation
        with torch.no_grad():
            response_tensors = ppo_model.generate(
                input_ids=prompt_tensors['input_ids'],
                attention_mask=prompt_tensors['attention_mask'],
                **generation_kwargs
            )

        response_texts = tokenizer.batch_decode(response_tensors, skip_special_tokens=True)
        rewards_list = compute_rouge_reward(response_texts, reference_texts)
        rewards_tensors = [torch.tensor(r, device=ppo_trainer.current_device, dtype=torch.float32) for r in rewards_list]

        # Prepare tensors
        list_query_tensors = [q.to(ppo_trainer.current_device) for q in prompt_tensors['input_ids']]
        list_response_tensors = [r.to(ppo_trainer.current_device) for r in response_tensors]

        if not rewards_tensors:
            continue

        # PPO step
        stats = ppo_trainer.step(list_query_tensors, list_response_tensors, rewards_tensors)
        success_count += 1

        mean_reward = torch.stack(rewards_tensors).mean().item()
        ppo_loss = stats.get('ppo/loss/total', 0)

        cumulative_rewards.append(mean_reward)
        avg_reward = np.mean(cumulative_rewards[-10:]) if len(cumulative_rewards) >= 10 else mean_reward

        # 🔰 CHỈ HIỂN THỊ THÔNG SỐ CƠ BẢN
        pbar.set_postfix({
            "reward": f"{mean_reward:.1f}",
            "loss": f"{ppo_loss:.1f}",
            "success": f"{success_count}/{batch_idx+1}"
        })

        # Chỉ in thông báo mỗi 10 batch
        if batch_idx % 10 == 0:
            print(f"Batch {batch_idx}: Reward: {mean_reward:.1f}, Loss: {ppo_loss:.1f}")

    except Exception as e:
        # Chỉ in lỗi nghiêm trọng
        if "CUDA" in str(e) or "memory" in str(e).lower():
            print(f"Lỗi batch {batch_idx}: {e}")
        torch.cuda.empty_cache()
        continue

print(f"\nHoàn tất: {success_count}/{total_batches} batch thành công")
print(f"Reward trung bình: {np.mean(cumulative_rewards):.1f}")

Bắt đầu huấn luyện PPO với 18000 mẫu...


PPO Training:   0%|          | 0/1125 [00:00<?, ?it/s]

ROUGE rougeL - Mean: 0.335, Max: 0.626, Min: 0.224
Reward (Linear 100) - Mean: 33.5, Max: 62.6


PPO Training:   0%|          | 1/1125 [00:07<2:13:55,  7.15s/it, reward=33.5, loss=30.7, success=1/1]

Batch 0: Reward: 33.5, Loss: 30.7
ROUGE rougeL - Mean: 0.338, Max: 0.652, Min: 0.217
Reward (Linear 100) - Mean: 33.8, Max: 65.2


...

ROUGE rougeL - Mean: 0.275, Max: 0.362, Min: 0.225
Reward (Linear 100) - Mean: 27.5, Max: 36.2


PPO Training: 100%|█████████▉| 1124/1125 [2:11:04<00:07,  7.07s/it, reward=27.5, loss=0.3, success=1124/1124]

ROUGE rougeL - Mean: 0.284, Max: 0.390, Min: 0.216
Reward (Linear 100) - Mean: 28.4, Max: 39.0


PPO Training: 100%|██████████| 1125/1125 [2:11:10<00:00,  7.00s/it, reward=28.4, loss=0.6, success=1125/1125]


Hoàn tất: 1125/1125 batch thành công
Reward trung bình: 28.8


In [26]:
def DemoPPO(prompt_text):
    """
    Chạy demo tóm tắt CHỈ DÙNG mô hình PPO đã load (Base + Adapter).
    """
    if tokenizer is None or ppo_model is None:
        print("LỖI: Tokenizer hoặc Model PPO chưa được load. Vui lòng chạy lại các ô code trước.")
        return None

    print(f"Input: {prompt_text}")
    full_prompt = "tóm tắt: " + prompt_text

    inputs = tokenizer(full_prompt, return_tensors="pt", max_length=512, truncation=True).to(device)

    print(f"--- Đang chạy demo với mô hình: PPO ---")

    with torch.no_grad():
        outputs = ppo_model.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=1,
            no_repeat_ngram_size=2
        )

    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    print(f"Output (PPO): {result}\n" + "-"*20)
    return result


In [30]:
import evaluate
from datasets import load_dataset
from tqdm import tqdm

def evaluate_rouge_l():
    """
    Đánh giá điểm ROUGE-L của mô hình PPO
    """
    print("🔍 Đang đánh giá ROUGE-L...")

    # Tải tập test
    test_dataset = load_dataset("nam194/vietnews", split="test[:50]")

    # Chuẩn bị metric
    rouge_metric = evaluate.load("rouge")

    all_predictions = []
    all_references = []

    # Chuyển mô hình sang chế độ đánh giá
    ppo_model.eval()

    print("🔄 Đang tạo tóm tắt...")
    for i in tqdm(range(len(test_dataset))):
        example = test_dataset[i]
        input_text = "tóm tắt: " + example["article"]
        reference_text = example["abstract"]

        # Tokenize
        inputs = tokenizer(input_text, return_tensors="pt", max_length=512, truncation=True).to(device)

        # Generate
        with torch.no_grad():
            outputs = ppo_model.generate(
                **inputs,
                max_new_tokens=128,
                num_beams=1,
                no_repeat_ngram_size=2,
                early_stopping=True
            )

        prediction = tokenizer.decode(outputs[0], skip_special_tokens=True)
        all_predictions.append(prediction)
        all_references.append(reference_text)

    # Tính ROUGE-L
    rouge_scores = rouge_metric.compute(
        predictions=all_predictions,
        references=all_references,
        use_aggregator=True
    )

    # Hiển thị kết quả
    print(f"\n🎯 ROUGE-L: {rouge_scores['rougeL'] * 100:.2f}")

    return rouge_scores['rougeL']

# Chạy đánh giá
rouge_l_score = evaluate_rouge_l()

🔍 Đang đánh giá ROUGE-L...
🔄 Đang tạo tóm tắt...


100%|██████████| 50/50 [07:50<00:00,  9.41s/it]



🎯 ROUGE-L: 29.19
